# Test de personalidad
## 01 · Datos y definición de perfiles

Proyecto de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

Queremos **predecir el perfil de personalidad de una persona** a partir de sus respuestas en tres formularios (personalidad, música y relaciones amorosas).

En este primer cuaderno hacemos tres cosas:
1. Cargar los datos.
2. Definir, con reglas claras y escritas, los perfiles que vamos a predecir.
3. Guardar un archivo con el perfil ya asignado para usarlo en los siguientes cuadernos (EDA y modelado).

## Datos

Usamos el archivo `Datos_Unidos.xlsx`, que ya pasó por la limpieza del proyecto:

- Cada fila es **una persona**, identificada con un código anónimo (`U001`, `U002`...). No hay correos, números de cuenta ni marcas de tiempo.
- Reúne los tres formularios. Todas las personas respondieron el de personalidad; algunas no respondieron el de relaciones amorosas.
- Las preguntas de personalidad empiezan con `pers_`, las de música con `mus_` y las de relaciones amorosas con `amor_`.
- La maestra autorizó cruzar la información de los formularios.

**Cómo subir el archivo en Colab:** al ejecutar la celda de carga, si el archivo no está, se abrirá una ventana para elegirlo. También puedes arrastrarlo al panel de archivos (ícono de carpeta, a la izquierda). Recuerda que **cada vez que se cierra la sesión de Colab los archivos subidos se borran**, así que hay que subirlo de nuevo.

### Preparar el entorno
*Qué hace:* importa las herramientas que vamos a usar y revisa si estamos en Google Colab.

In [ ]:
import os
import re
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print("pandas", pd.__version__, "| ejecutando en Colab:", EN_COLAB)

### Cargar los datos
*Qué hace:* busca el archivo `Datos_Unidos.xlsx` (si no está, te pide subirlo), lo lee y cuenta cuántas personas hay.

In [ ]:
ARCHIVO = "Datos_Unidos.xlsx"

if not os.path.exists(ARCHIVO):
    if EN_COLAB:
        print("Elige el archivo Datos_Unidos.xlsx en la ventana que se abre...")
        files.upload()
    if not os.path.exists(ARCHIVO):
        raise FileNotFoundError(
            f"No se encontró '{ARCHIVO}'. Súbelo al panel de archivos de Colab con ese nombre exacto."
        )

df = pd.read_excel(ARCHIVO, sheet_name="DatosUnidos")

print(f"Personas (filas): {df.shape[0]} | Variables (columnas): {df.shape[1]}")
print("Con formulario de música:", (df["tiene_musica"] == "Sí").sum())
print("Con formulario de relaciones amorosas:", (df["tiene_amor"] == "Sí").sum())

### Un vistazo a las preguntas que definen el perfil
*Qué hace:* muestra las primeras filas de las dos preguntas que usaremos para construir los perfiles:

- **pers_q06:** "Cuando tienes tiempo libre, ¿prefieres pasar tiempo solo o acompañado? ¿Por qué?"
- **pers_q07:** "¿Cómo sueles comportarte cuando estás con un grupo de personas que no conoces?"

In [ ]:
display(df[["id_persona", "pers_genero", "pers_q06", "pers_q07"]].head())

## Cómo definimos los perfiles

El formulario **no trae** una pregunta que diga "eres introvertido" o "eres extrovertido", así que construimos la etiqueta nosotros con **reglas escritas**. Las reglas usan solo expresiones regulares, sin IA ni NLP, y cualquiera puede repetirlas y obtener lo mismo.

**Paso 1: clasificar cada respuesta**

- **Pregunta 6 (solo o acompañado):** buscamos la *primera* palabra clave que aparece.
  - `solo` o `sola` → **solo**
  - `acompañado` o `acompañada` → **acompanado**
  - `depende`, `ambas` o `ambos` → **mixto**
- **Pregunta 7 (grupo de desconocidos):** buscamos palabras clave.
  - **reservado:** reservado, callado, tímido, introvertido, incomodidad, nerviosismo, vergüenza, "no hablo", "escucho", "observo"
  - **sociable:** sociable, extrovertido, amable, gracioso
  - Si aparecen palabras de las dos listas → **senales_mixtas**
  - Si no aparece ninguna → **sin_clasificar** (no inventamos una interpretación)

**Importante:** estas dos preguntas se usan para crear la etiqueta, así que **no se usarán como variables para predecir** en el modelado. Si no, el modelo solo repetiría nuestra regla y el resultado sería trampa.

### Funciones de clasificación
*Qué hace:* define las reglas anteriores como funciones y las aplica a cada persona.

In [ ]:
def normalizar(texto):
    """Pasa a minúsculas y quita acentos, para que las reglas no dependan de cómo se escribió."""
    t = unicodedata.normalize("NFKD", str(texto))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.lower()


def clasificar_q6(texto):
    if pd.isna(texto):
        return "sin_dato"
    m = re.search(r"\b(sol[oa]|acompan\w+|depende|ambas|ambos)\b", normalizar(texto))
    if not m:
        return "sin_clasificar"
    palabra = m.group(1)
    if palabra.startswith("sol"):
        return "solo"
    if palabra.startswith("acompan"):
        return "acompanado"
    return "mixto"


RESERVADO = re.compile(
    r"reservad|callad|timid|introvert|incomod|nervios|verguenza|no hablo|no hablar|no suelo hablar|escucho|observ"
)
SOCIABLE = re.compile(r"sociable|extrovert|amable|gracios")


def clasificar_q7(texto):
    if pd.isna(texto):
        return "sin_dato"
    t = normalizar(texto)
    reservado, sociable = bool(RESERVADO.search(t)), bool(SOCIABLE.search(t))
    if reservado and sociable:
        return "senales_mixtas"
    if reservado:
        return "reservado"
    if sociable:
        return "sociable"
    return "sin_clasificar"


df["q6_clase"] = df["pers_q06"].map(clasificar_q6)
df["q7_clase"] = df["pers_q07"].map(clasificar_q7)

display(pd.crosstab(df["q6_clase"], df["q7_clase"], margins=True))

### Revisar la clasificación persona por persona
*Qué hace:* muestra, para cada persona, la respuesta (recortada) y la clase que le asignó la regla. **Hay que leerla con calma** para confirmar que tenga sentido; algunas decisiones dependen de las palabras clave que elegimos (por ejemplo, "nerviosismo", "incomodidad" o "escucho" se tomaron como señal de ser reservado).

In [ ]:
revision = df[["id_persona", "pers_q06", "q6_clase", "pers_q07", "q7_clase"]].copy()
revision["pers_q06"] = revision["pers_q06"].astype(str).str.slice(0, 55)
revision["pers_q07"] = revision["pers_q07"].astype(str).str.slice(0, 55)
display(revision)

## ¿Cuántos perfiles se pueden predecir?

Primero probamos con **3 perfiles** (introvertido, ambivertido, extrovertido). Sumamos puntos a cada respuesta:

- Pregunta 6: solo = 0, mixto = 1, acompañado = 2.
- Pregunta 7: reservado = 0, sin clasificar o señales mixtas = 1 (sin evidencia, cuenta como neutral), sociable = 2.
- Total de 0 a 1 → Introvertido · 2 → Ambivertido · 3 o más → Extrovertido.

*Qué hace la celda siguiente:* cuenta cuántas personas quedan en cada perfil con este esquema.

In [ ]:
PUNTOS_Q6 = {"solo": 0, "mixto": 1, "acompanado": 2}
PUNTOS_Q7 = {"reservado": 0, "sin_clasificar": 1, "senales_mixtas": 1, "sociable": 2}

puntaje = df["q6_clase"].map(PUNTOS_Q6) + df["q7_clase"].map(PUNTOS_Q7)


def a_tres_perfiles(p):
    if pd.isna(p):
        return "sin_dato"
    if p <= 1:
        return "Introvertido"
    if p == 2:
        return "Ambivertido"
    return "Extrovertido"


prueba_3_perfiles = puntaje.map(a_tres_perfiles)
print(prueba_3_perfiles.value_counts())

**Resultado:** con 3 perfiles, solo **una persona** sale extrovertida. Un modelo no puede aprender a reconocer un perfil con un solo ejemplo, y tampoco se puede evaluar. Es un resultado sobre nuestros datos: el grupo es mayoritariamente introvertido.

Por eso usamos **2 perfiles**:

- **Introvertido:** prefiere estar solo (pregunta 6) **y** se describe reservado con desconocidos (pregunta 7).
- **No introvertido:** muestra al menos una señal clara de sociabilidad: prefiere estar acompañado o depende (pregunta 6), o se describe sociable (pregunta 7). Este grupo reúne a las personas ambivertidas y extrovertidas.
- **Por revisar:** prefiere estar solo, pero su respuesta de la pregunta 7 no se puede clasificar con las reglas. **No se fuerza una etiqueta**; el equipo decide.

### Asignar el perfil
*Qué hace:* aplica las reglas de 2 perfiles. Si el equipo decide qué hacer con las personas "Por revisar", se anota en `DECISION_EQUIPO` (por ejemplo `{"U006": "Introvertido"}`) y queda registrado en la columna `perfil_origen`.

In [ ]:
PERFILES_VALIDOS = ("Introvertido", "No introvertido")


def asignar_perfil(q6, q7):
    if q6 == "solo" and q7 == "reservado":
        return "Introvertido"
    if q6 in ("acompanado", "mixto") or q7 == "sociable":
        return "No introvertido"
    return "Por revisar"


df["perfil"] = [asignar_perfil(a, b) for a, b in zip(df["q6_clase"], df["q7_clase"])]
df["perfil_origen"] = "regla"

# Decisiones del equipo sobre los casos "Por revisar". Déjalo vacío si aún no se decide.
DECISION_EQUIPO = {}

for persona, nuevo_perfil in DECISION_EQUIPO.items():
    fila = df["id_persona"] == persona
    assert fila.any(), f"{persona} no existe en los datos"
    assert (df.loc[fila, "perfil"] == "Por revisar").all(), f"{persona} no estaba 'Por revisar'"
    assert nuevo_perfil in PERFILES_VALIDOS, f"Perfil no válido: {nuevo_perfil}"
    df.loc[fila, "perfil"] = nuevo_perfil
    df.loc[fila, "perfil_origen"] = "decision_equipo"

print(df["perfil"].value_counts())
print("\nPersonas por revisar:", list(df.loc[df["perfil"] == "Por revisar", "id_persona"]))

### Distribución de los perfiles
*Qué hace:* dibuja cuántas personas hay en cada perfil.

In [ ]:
conteo = df["perfil"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(conteo.index, conteo.values)
plt.title("Personas por perfil")
plt.ylabel("Personas")
plt.show()

### Guardar el archivo con el perfil
*Qué hace:* guarda `Datos_Con_Perfil.xlsx` con todos los datos más el perfil asignado, y una segunda hoja con las reglas usadas. Los siguientes cuadernos (EDA y modelado) empiezan desde este archivo. En Colab también se descarga a tu computadora.

In [ ]:
reglas = pd.DataFrame({
    "elemento": [
        "q6_clase", "q7_clase", "Introvertido", "No introvertido", "Por revisar", "Variables prohibidas como predictoras",
    ],
    "regla": [
        "Primera palabra clave en la pregunta 6: solo/sola, acompanado/a, depende/ambas/ambos",
        "Palabras clave en la pregunta 7: reservado (callado, timido, introvertido, incomodidad, nerviosismo, escucho...) o sociable (sociable, extrovertido, amable, gracioso)",
        "q6_clase = solo Y q7_clase = reservado",
        "q6_clase = acompanado o mixto, O q7_clase = sociable",
        "Cualquier otro caso (no se fuerza una etiqueta)",
        "pers_q06 y pers_q07 (con ellas se creó la etiqueta; usarlas para predecir sería trampa)",
    ],
})

SALIDA = "Datos_Con_Perfil.xlsx"
with pd.ExcelWriter(SALIDA) as escritor:
    df.to_excel(escritor, sheet_name="DatosConPerfil", index=False)
    reglas.to_excel(escritor, sheet_name="ReglasPerfil", index=False)

print("Guardado:", SALIDA, "| filas:", len(df), "| columnas:", df.shape[1])

if EN_COLAB:
    files.download(SALIDA)

## Conclusiones

- Quedan **23 personas**. Con las reglas, **14 son Introvertidas**, **7 No introvertidas** y **2 están Por revisar**.
- Con 3 perfiles solo una persona sale extrovertida, así que no es viable. Se usan **2 perfiles**.
- Los perfiles están desbalanceados (hay más introvertidos). En el modelado no basta mirar el porcentaje de aciertos, porque un modelo que siempre diga "Introvertido" ya acertaría mucho sin aprender nada.
- Con tan pocas personas, los resultados del modelado serán **exploratorios**, y así los presentaremos.
- Las preguntas 6 y 7 **no** se usarán como variables para predecir.

**Siguiente cuaderno:** `02_EDA`, el análisis exploratorio de los datos.